<a href="https://colab.research.google.com/github/EugeneOny1/-civil-infrastructure-monitoring---CS-PROJECT-II/blob/feature-branch/02_data_inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Purpose: Inspect the actual datasets before annotation validation, conversion and training.

This notebook doesn't modify Raw Data. It reports image count, annotation formats, image integrity, dimensions, sample images and te observed RDD2022 YOLO Labels

##Target Classes


1.   Crack
2.   Pothole
3.   Surface deterioration



In [15]:
from pathlib import Path
from collections import Counter
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image


PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")
DATA_DIR = PROJECT_DIR / "Data"

RAW_DIR = DATA_DIR / "Raw Data"
SELECTED_DIR = DATA_DIR / "Selected Data"
ANNOTATED_DIR = DATA_DIR / "Annotated Data"
PROCESSED_DIR = DATA_DIR / "Processed Data"

SDNET_DIR = RAW_DIR / "SDNET2018"
RDD_DIR = RAW_DIR / "RDD2022"
KENYAN_DIR = RAW_DIR / "Kenyan_Data"

SDNET_ANNOTATED = ANNOTATED_DIR / "SDNET2018"
RDD_ANNOTATED = ANNOTATED_DIR / "RDD2022"
KENYAN_ANNOTATED = ANNOTATED_DIR / "Kenyan_Data"

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR.exists())
print("Selected data:", SELECTED_DIR.exists())
print("Annotated data:", ANNOTATED_DIR.exists())
print("Processed data:", PROCESSED_DIR.exists())

Project: /content/drive/MyDrive/CS PROJECT II
Raw data: True
Selected data: True
Annotated data: True
Processed data: True


In [16]:
from pathlib import Path
from collections import Counter

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

DATASET_PATHS = {
    "SDNET2018": SDNET_DIR,
    "RDD2022": RDD_DIR,
    "Kenyan_Data": KENYAN_DIR,
}

for name, path in DATASET_PATHS.items():
    print("=" * 70)
    print(name)
    print("Expected path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    files = [p for p in path.rglob("*") if p.is_file()]

    print("Total files:", len(files))

    extensions = Counter(
        p.suffix.lower() if p.suffix else "[none]"
        for p in files
    )

    print("File extensions:")
    for ext, count in extensions.most_common():
        print(f"  {ext}: {count}")

    images = [
        p for p in files
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    print("Images detected:", len(images))

    print("\nSample files:")
    for p in files[:10]:
        print(" ", p)

SDNET2018
Expected path: /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018
Exists: True
Total files: 56102
File extensions:
  .jpg: 56102
Images detected: 56102

Sample files:
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Cracked/071-62.jpg
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Cracked/071-44.jpg
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Cracked/071-45.jpg
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Cracked/071-56.jpg
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Cracked/071-52.jpg
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Cracked/071-51.jpg
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Cracked/071-46.jpg
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Cracked/071-59.jpg
  /content/drive/MyDrive/CS PROJECT II/Data/Raw Data/SDNET2018/Pavements/Crac

## 1. Dataset Overview

The counts below are taken from the directories currently present in the Google Drive. They are not hard coded

In [17]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG", ".WEBP"}
ANNOTATION_EXTENSIONS = {".xml", ".txt", ".json"}

def files_with_extensions(directory, extensions):
    directory = Path(directory)
    if not directory.exists():
        return []
    return [p for p in directory.rglob("*") if p.is_file() and p.suffix in extensions]

def count_images(directory):
    return len(files_with_extensions(directory, IMAGE_EXTENSIONS))

def count_files(directory, extensions):
    return len(files_with_extensions(directory, extensions))

datasets = {
    "SDNET2018": SDNET_DIR,
    "RDD2022": RDD_DIR,
    "Kenyan_Data": KENYAN_DIR,
}

overview = []
for name, path in datasets.items():
    overview.append({
        "dataset": name,
        "images": count_images(path),
        "xml": count_files(path, {".xml"}),
        "yolo_txt": count_files(path, {".txt"}),
        "json": count_files(path, {".json"}),
    })

overview_df = pd.DataFrame(overview)
display(overview_df)

,dataset,images,xml,yolo_txt,json
0,SDNET2018,56102,0,0,0
1,RDD2022,36984,0,38386,0
2,Kenyan_Data,297,0,0,0


## 2. Image Integrity Checks

A readable image is required before it can enter annotation validation or training

In [ ]:
def check_image_integrity(directory, max_images=None):
    image_paths = files_with_extensions(directory, IMAGE_EXTENSIONS)
    if max_images is not None:
        image_paths = image_paths[:max_images]

    valid = 0
    invalid = []
    for path in image_paths:
        try:
            with Image.open(path) as img:
                img.verify()
            valid += 1
        except Exception as exc:
            invalid.append((str(path), str(exc)))

    return len(image_paths), valid, invalid

integrity_rows = []
for name, path in datasets.items():
    total, valid, invalid = check_image_integrity(path)
    integrity_rows.append({
        "dataset": name,
        "checked": total,
        "valid": valid,
        "invalid": len(invalid)
    })

integrity_df = pd.DataFrame(integrity_rows)
display(integrity_df)

In [ ]:
# Show a few corrupted/unreadable files if any were found.
for name, path in datasets.items():
    _, _, invalid = check_image_integrity(path)
    if invalid:
        print(f"\n{name} invalid examples:")
        for item in invalid[:10]:
            print(item)

## 3. Image Dimensions

Dimension statistics help identify unusual files and dataset difference. They are descriptive only: no resizing is performed here.

In [ ]:
def image_dimension_summary(directory, max_images=None):
    image_paths = files_with_extensions(directory, IMAGE_EXTENSIONS)
    if max_images is not None:
        image_paths = image_paths[:max_images]

    rows = []
    for path in image_paths:
        try:
            with Image.open(path) as img:
                w, h = img.size
            rows.append({"path": str(path), "width": w, "height": h, "aspect_ratio": round(w / h, 4)})
        except Exception:
            pass

    return pd.DataFrame(rows)

dimension_summaries = {}

for name, path in datasets.items():
    df = image_dimension_summary(path)
    dimension_summaries[name] = df

    if not df.empty:
        print(f"\n{name}")
        print("Images measured:", len(df))
        print("Width:", df["width"].min(), "to", df["width"].max())
        print("Height:", df["height"].min(), "to", df["height"].max())
        print("Mean aspect ratio:", round(df["aspect_ratio"].mean(), 3))

## 4. Random Image Visualization

In [ ]:
def random_image_paths(directory, n=4, seed=42):
    paths = files_with_extensions(directory, IMAGE_EXTENSIONS)
    rng = random.Random(seed)
    return rng.sample(paths, min(n, len(paths)))

for name, path in datasets.items():
    samples = random_image_paths(path, n=3, seed=42)

    if not samples:
        continue

    fig, axes = plt.subplots(1, len(samples), figsize=(15, 5))
    if len(samples) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, samples):
        try:
            with Image.open(img_path) as img:
                ax.imshow(img.convert("RGB"))
            ax.set_title(img_path.name, fontsize=8)
            ax.axis("off")
        except Exception as exc:
            ax.set_title(f"Unreadable\n{exc}")
            ax.axis("off")

    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

## 5. RDD2022 YOLO Annotation Inspection

Notebook 01 established that the current RDD2022 copy contains YOLO .txt files and no XML files.

This section inspects the actual class IDs and annotation syntax. Do not finalize the class mapping until the output below has been checked.

In [ ]:
RDD_TXT_FILES = files_with_extensions(RDD_DIR, {".txt"})

def read_yolo_lines(txt_path):
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        return [line.strip() for line in f if line.strip()]

class_id_counts = Counter()
invalid_line_examples = []

for txt_path in RDD_TXT_FILES:
    for line in read_yolo_lines(txt_path):
        parts = line.split()
        if len(parts) != 5:
            if len(invalid_line_examples) < 20:
                invalid_line_examples.append((str(txt_path), line))
            continue

        try:
            class_id = int(parts[0])
            values = [float(v) for v in parts[1:]]
            class_id_counts[class_id] += 1
        except ValueError:
            if len(invalid_line_examples) < 20:
                invalid_line_examples.append((str(txt_path), line))

print("Observed RDD2022 class IDs:")
print(dict(sorted(class_id_counts.items())))

if invalid_line_examples:
    print("\nInvalid YOLO line examples:")
    for item in invalid_line_examples[:20]:
        print(item)

In [ ]:
# Show actual sample YOLO annotation files.
for txt_path in RDD_TXT_FILES[:5]:
    print("\nFILE:", txt_path)
    for line in read_yolo_lines(txt_path)[:10]:
        print(line)

## RDD mapping checkpoint

For standard RDD four-class road-damage taxonomy, the expected mapping is:

*   D00 - Crack
*   D10 - Crack
*   D20 - Crack
*   D40 - Pothole

If the obsevered class IDs are not the expected four IDs, STOP before Notebook 03 and update the mapping from the actual annotation files.

Do not use D43 or D44 as RDD2022 classes unless your actual downloaded dataset demonstrably contains those classes. They are part of the four-class RDD2022 mappinf being used here





## 6. Basic Image-Annotation Pairing Check for RDD2022

In [ ]:
# Check RDD2022 image–annotation pairing using the full relative path.
# This avoids false matches when different subfolders contain files
# with the same filename stem.

RDD_PATH = RDD_DIR

rdd_images = files_with_extensions(RDD_PATH, IMAGE_EXTENSIONS)
rdd_labels = files_with_extensions(RDD_PATH, {".txt"})


def relative_stem(path, root):
    return path.relative_to(root).with_suffix("").as_posix()


image_keys = {
    relative_stem(path, RDD_PATH)
    for path in rdd_images
}

label_keys = {
    relative_stem(path, RDD_PATH)
    for path in rdd_labels
}


paired_keys = image_keys & label_keys
missing_images = sorted(label_keys - image_keys)
missing_labels = sorted(image_keys - label_keys)


print("RDD2022 Image–Annotation Pairing")
print("=" * 60)

print("Images:", len(image_keys))
print("YOLO annotations:", len(label_keys))
print("Paired image/annotation files:", len(paired_keys))
print("Annotations without matching image:", len(missing_images))
print("Images without matching annotation:", len(missing_labels))

print("\nExamples of annotations without images:")
for item in missing_images[:10]:
    print(" ", item)

print("\nExamples of images without annotations:")
for item in missing_labels[:10]:
    print(" ", item)

In [ ]:
RDD_PATH = DATASET_PATHS["RDD2022"]

image_files = {
    p.relative_to(RDD_PATH).with_suffix("").as_posix()
    for p in RDD_PATH.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
}

label_files = {
    p.relative_to(RDD_PATH).with_suffix("").as_posix()
    for p in RDD_PATH.rglob("*.txt")
}

missing_images = sorted(label_files - image_files)
missing_labels = sorted(image_files - label_files)

print("RDD2022 Image/Label Pairing")
print("-" * 50)

print("Images:", len(image_files))
print("YOLO labels:", len(label_files))
print("Labels without matching image:", len(missing_images))
print("Images without matching label:", len(missing_labels))

print("\nExamples of labels without images:")
for x in missing_images[:20]:
    print(" ", x)

print("\nExamples of images without labels:")
for x in missing_labels[:20]:
    print(" ", x)

## 7. Dataset Inspection Summary

In [ ]:
print("DATASET SUMMARY")
print("=" * 60)
display(overview_df)

print("\nNEXT STAGE")
print("Notebook 03 will validate annotations, apply the target class mapping,")
print("construct a normalized manifest, and create leakage-aware dataset splits.")